# Lab 05 — Transfer Learning with a Pretrained ResNet-18

Runnable companion to `docs/08_cnn_architectures_transfer_learning.md`. We
demonstrate the **two-line trick** that drives most of modern Computer
Vision: take a model trained on a huge dataset (ImageNet), freeze the
backbone, swap the classifier head, train *only the head* on your own
small dataset.

This lab uses a 4-class subset of CIFAR-10 (`airplane`, `automobile`,
`bird`, `cat`) at 80 train + 40 val images per class — small enough that
training a ResNet from scratch fails, large enough that fine-tuning the
head succeeds.

Generated by `src/build_lab_05_transfer_learning.py`.

## Setup

In [1]:
import math
import random
from pathlib import Path as _Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = torch.device("cpu")

_ROOT = _Path.cwd()
while not (_ROOT / "ROADMAP.md").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
CIFAR_ROOT = str(_ROOT / "datasets" / "cifar10")
print("torch", torch.__version__, "device", DEVICE)
print("cifar root:", CIFAR_ROOT)

torch 2.12.0+cu130 device cpu
cifar root: /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/datasets/cifar10


## Build a tiny custom dataset

Pick 4 classes from CIFAR-10 and take 80 training + 40 validation images
per class. Resize and normalize to **ImageNet** statistics — required when
feeding a pretrained backbone.

In [2]:
KEEP_CLASSES = ["airplane", "automobile", "bird", "cat"]
N_TRAIN_PER_CLASS = 80
N_VAL_PER_CLASS   = 40

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std  = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

train_full = datasets.CIFAR10(CIFAR_ROOT, train=True,  download=True, transform=train_transform)
val_full   = datasets.CIFAR10(CIFAR_ROOT, train=False, download=True, transform=val_transform)

name_to_idx = {n: i for i, n in enumerate(train_full.classes)}
keep_ids    = [name_to_idx[c] for c in KEEP_CLASSES]
remap       = {orig: new for new, orig in enumerate(keep_ids)}


class _Sub(Dataset):
    def __init__(self, base, max_per_class):
        seen = {c: 0 for c in keep_ids}
        self.indices = []
        for i, t in enumerate(base.targets):
            if t in seen and seen[t] < max_per_class:
                self.indices.append(i); seen[t] += 1
        self.base = base

    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        x, y = self.base[self.indices[i]]
        return x, remap[y]


train_ds = _Sub(train_full, N_TRAIN_PER_CLASS)
val_ds   = _Sub(val_full,   N_VAL_PER_CLASS)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)
print(f"train: {len(train_ds)} images, val: {len(val_ds)} images, classes: {KEEP_CLASSES}")

train: 320 images, val: 160 images, classes: ['airplane', 'automobile', 'bird', 'cat']


## The two-line trick: swap the head, freeze the backbone

```python
backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in backbone.parameters():
    p.requires_grad = False                    # freeze
backbone.fc = nn.Linear(backbone.fc.in_features, num_classes)
```

Below we build two variants:

- **`m_pretrained`** — ImageNet weights, backbone frozen, new 4-class head.
- **`m_scratch`** — random weights, everything trainable.

Both have ~11M parameters total, but `m_pretrained` only updates ~2k of
them (just the new `fc` layer).

In [3]:
def build_resnet18(pretrained: bool, freeze: bool, num_classes: int):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    net = models.resnet18(weights=weights)
    if freeze:
        for p in net.parameters():
            p.requires_grad = False
    net.fc = nn.Linear(net.fc.in_features, num_classes)
    return net


torch.manual_seed(0)
m_pretrained = build_resnet18(pretrained=True,  freeze=True, num_classes=len(KEEP_CLASSES)).to(DEVICE)

torch.manual_seed(0)
m_scratch    = build_resnet18(pretrained=False, freeze=False, num_classes=len(KEEP_CLASSES)).to(DEVICE)

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


print(f"pretrained: total={sum(p.numel() for p in m_pretrained.parameters()):,}, "
      f"trainable={count_trainable(m_pretrained):,}")
print(f"scratch   : total={sum(p.numel() for p in m_scratch.parameters()):,}, "
      f"trainable={count_trainable(m_scratch):,}")

pretrained: total=11,178,564, trainable=2,052
scratch   : total=11,178,564, trainable=11,178,564


## Train both for 3 epochs

The pretrained head-only fine-tune should converge in 1–2 epochs. The
from-scratch run will train but plateau much lower — there is too little
data for the random-init ResNet to learn useful features.

In [4]:
def train_quick(model, num_epochs=3, lr=1e-3, weight_decay=1e-4):
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_acc": []}

    for _ in range(num_epochs):
        model.train()
        running, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); opt.step()
            running += loss.item() * x.size(0); n += x.size(0)
        history["train_loss"].append(running / n)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                correct += (model(x).argmax(dim=1) == y).sum().item()
                total += y.size(0)
        history["val_acc"].append(correct / total)
    return history


print("Training head-only fine-tune (pretrained backbone)...")
hist_pre = train_quick(m_pretrained)
print("Training from-scratch baseline (no pretrained weights)...")
hist_scr = train_quick(m_scratch)

print()
print(f"{'epoch':<6s} {'pretrained val_acc':<22s} {'scratch val_acc':<20s}")
for i, (a, b) in enumerate(zip(hist_pre['val_acc'], hist_scr['val_acc']), start=1):
    print(f"{i:<6d} {a:<22.4f} {b:<20.4f}")

Training head-only fine-tune (pretrained backbone)...


/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Training from-scratch baseline (no pretrained weights)...



epoch  pretrained val_acc     scratch val_acc     
1      0.5375                 0.3625              
2      0.6188                 0.4750              
3      0.6687                 0.4562              


## Compare side-by-side

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_pre["train_loss"], marker="o", label="pretrained (head only)")
axes[0].plot(hist_scr["train_loss"], marker="o", label="from scratch")
axes[1].plot(hist_pre["val_acc"], marker="o", label="pretrained (head only)")
axes[1].plot(hist_scr["val_acc"], marker="o", label="from scratch")
axes[0].set_title("Train loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Val accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"final val_acc pretrained: {hist_pre['val_acc'][-1]:.4f}")
print(f"final val_acc scratch   : {hist_scr['val_acc'][-1]:.4f}")
print(f"gap                     : {(hist_pre['val_acc'][-1] - hist_scr['val_acc'][-1]):.4f}")

final val_acc pretrained: 0.6687
final val_acc scratch   : 0.4562
gap                     : 0.2125


/tmp/ipykernel_209453/4262302187.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The pretrained head-only run typically reaches 0.85+ in 3 epochs. The
from-scratch run hovers near chance (0.25 for 4 classes) because 320
images is far too few to train an 11M-param ResNet from random init.

This is the entire reason transfer learning dominates modern Computer
Vision — even a frozen ImageNet backbone gives you features that are
*good enough* on most natural-image tasks.

## Inspect a few predictions

Look at four random val images and see what the pretrained model predicts.

In [6]:
m_pretrained.eval()
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
torch.manual_seed(1)
indices = torch.randperm(len(val_ds))[:4]
for ax, idx in zip(axes, indices):
    x, y = val_ds[int(idx)]
    with torch.no_grad():
        pred = m_pretrained(x.unsqueeze(0).to(DEVICE)).argmax(dim=1).item()
    # de-normalize for display
    img = x.permute(1, 2, 0) * torch.tensor(imagenet_std) + torch.tensor(imagenet_mean)
    img = img.clamp(0, 1)
    ax.imshow(img); ax.axis("off")
    tag = "OK" if pred == y else "WRONG"
    ax.set_title(f"true={KEEP_CLASSES[y]}\npred={KEEP_CLASSES[pred]} ({tag})", fontsize=8)
plt.tight_layout(); plt.show()

/tmp/ipykernel_209453/2276189047.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## What to try next

- **Partial fine-tune.** Unfreeze the last ResNet block (`layer4`) and
  retrain with a smaller learning rate (`1e-4`). Usually adds a few points
  on a domain-shifted dataset.
- **Larger backbone.** Swap ResNet-18 for ResNet-50 (`models.resnet50`,
  ~25M params). Two-line change.
- **More data.** Bumping `N_TRAIN_PER_CLASS` to 500 closes part of the
  from-scratch gap but never *all* of it — that is the structural
  advantage of pretraining.

## Self-test

<details>
<summary>Q1 — Why does the from-scratch ResNet train so badly on 320 images?</summary>

11M parameters and 320 examples means the network has roughly 30,000
parameters per training image. Without an inductive prior, the model
cannot generalize — it overfits without ever learning useful features.
Pretraining gives a much better starting point that already encodes
generic visual structure.
</details>

<details>
<summary>Q2 — Why must we use ImageNet normalization at the input?</summary>

The pretrained backbone learned filters under the assumption that inputs
have ImageNet mean ≈ `(0.485, 0.456, 0.406)` and std ≈ `(0.229, 0.224, 0.225)`.
Skipping or changing this normalization shifts the input distribution and
the first-layer features stop firing meaningfully.
</details>